# Axe 1 : Vie personnelle 

Dans ce notebook, j'étudie les variables liées aux caractéristiques personnelles des employés afin d'évaluer leur impact sur l'attrition.

Ce notebook a pour objectif :
- d'explorer les variables de l'axe,
- de comparer avec l'attrition
- d'explorer des relations internes à l’axe
- de visualiser
- retenir seulement ce qui ressort clairement
---

## 1. Exploration des variables de l’axe
---

### 1.1 Import des librairies & chargement des données
---

In [1]:
# import librairies 

import pandas as pd
import pyarrow
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [ ]:
# chargement du df
df_clean = pd.read_parquet("C:/Users/Kemu/Documents/Formation/Projet Pro Data/2_Projets_Tests/projet-RH_test/data/processed/employees_clean.parquet")
df_clean.info()

### 1.2 Creation du DataFrame vie_perso
---
#### Colonnes incluses dans l'axe "Vie personnelle"

Cet axe regroupe les caractéristiques individuelles :
- âge  
- situation familiale  
- genre  
- niveau d'éducation  et domaines
- mobilité géographique  
donnant un contexte personnel pouvant influencer l'attrition.

In [ ]:
# colonnes vie_perso
col_perso = ['Age', 'DistanceFromHome','Education','EducationField','Gender','MaritalStatus','NumCompaniesWorked']

# creation df_vie_perso
df_vie_perso = df_clean[col_perso + ['Attrition']].copy()
df_vie_perso

#### 1.2.1 Création des tranches d’âge

L'âge est une variable continue peu lisible en l'état.  
Pour faciliter l'analyse et observer des tendances plus nettes, je regroupe les employés en classes d'âge cohérentes du point de vue RH :

- **18–25 ans** : début de carrière, forte mobilité
- **26–35 ans** : stabilisation professionnelle, construction familiale
- **36–45 ans** : maturité professionnelle
- **46–55 ans** : employés expérimentés
- **55+ ans** : fin de carrière

Ces tranches permettront d'étudier plus clairement la relation entre âge et attrition.

In [ ]:
# Creation tranche d'age
df_vie_perso["Tranche_Age"] = pd.cut(
    df_vie_perso["Age"],
    bins=[0 , 25, 35, 45, 55, 150],
    labels=["18-25", "26-35", "36-45", "46-55", "+55"],
    ordered=True
)

df_vie_perso["Tranche_Age"].value_counts()

#### 1.2.2 Regroupement distance domicile–travail

Pour faciliter l'analyse exploratoire et rendre les visualisations plus lisibles, la distance est regroupée en tranches cohérentes :

- **1–5 km** : très proche  
- **6–10 km** : proche  
- **11–20 km** : distance moyenne  
- **> 20 km** : éloigné  

Ces catégories permettent d’identifier plus facilement d’éventuelles relations entre la distance domicile–travail et l’attrition.


In [ ]:
# Creation des regroupement

df_vie_perso['Tranche_Distance'] = pd.cut(
    df_vie_perso['DistanceFromHome'],
    bins = [0, 5, 10, 20, 30],
    labels= ["1-5", "6-10", "11-20", ">20"],
    include_lowest=True
)

df_vie_perso["Tranche_Distance"].value_counts()

#### 1.2.3 Renommage des niveaux d’éducation

Les niveaux d’éducation sont initialement codés sous forme numérique (1 à 4).Pour faciliter la lecture et l’analyse, je remplace ces codes par des catégories explicites 
Cette transformation permet d’interpréter facilement la répartition des employés selon leur niveau d’étude.


In [ ]:
niveau_education = {
    1 : 'Below College',
    2 : 'College',
    3 : 'Bachelor',
    4 : 'Master',
    5 : 'Doctor'
    }

df_vie_perso["NiveauEducation"] = df_vie_perso['Education'].replace(niveau_education)
df_vie_perso

#### 1.2.4 Renommage des genres
La variable `Gender` utilise des codes bruts (`Male`, `Female`).  
Je les remplace par des libellés plus explicites (*Homme* et *Femme*) afin d'améliorer la lisibilité et la cohérence du jeu de données.

In [ ]:
df_vie_perso["Gender"] = df_vie_perso["Gender"].replace({
    "Male": "Homme",
    "Female": "Femme"
})
df_vie_perso

### 1.3  Analyse individuelle des variables 
---
Dans cette section, j’analyse plusieurs caractéristiques personnelles des employés afin de mieux comprendre la composition de la population étudiée.

#### 1.3.1 Tranche d’âge 
---
Je trie les tranches d’âge selon leur fréquence afin d’identifier rapidement les groupes les plus représentés au sein de l’entreprise et préparer l’analyse exploratoire.


##### Tableau des proportions

In [ ]:

age = df_vie_perso["Tranche_Age"].value_counts(normalize=True).sort_values(ascending=False)*100

df_age = age.reset_index()
df_age.columns = ["Tranche_Age" , 'Pourcentage']
df_age

##### Histogramme des tranches d'âge

Ce graphique montre la répartition des employés selon les tranches d'âge.  
On observe deux groupes majoritaires :

- **26–35 ans**  
- **36–45 ans**

Ils représentent la majorité de la population active de l’entreprise.

Les tranches :

- **18–25 ans** et **46–55 ans** sont moins représentées,  
- tandis que les **55+** sont très minoritaires.


In [ ]:


plt.figure(figsize=(7,5))

axe = plt.gca()
axe.spines['top'].set_visible(False)
axe.spines["right"].set_visible(False)

sns.barplot(data=df_age,y="Pourcentage", x="Tranche_Age",color="skyblue")
plt.xlabel("Tranche d'Age")
plt.ylabel("Pourcentage")
plt.title("Répartition des employéess par tranche d'age (en %)")
plt.tight_layout()
plt.show(),

Après avoir examiné la répartition par âge, je m’intéresse maintenant à la proximité des employés avec leur lieu de travail à travers la variable `DistanceFromHome`.

#### 1.3.2 DistanceFromHome
---
J’analyse ici la répartition des employés selon la distance entre leur domicile et le lieu de travail, afin d’identifier les niveaux de proximité les plus fréquents dans l’entreprise.


##### Tableau des proportions

In [ ]:

distanceFromHome = df_vie_perso["Tranche_Distance"].value_counts(normalize=True).sort_values(ascending=False)*100

df_distanceFromHome = distanceFromHome.reset_index()
df_distanceFromHome.columns = ["Tranche_Distance", "Pourcentage"]

df_distanceFromHome

#### 

#### Répartition des employés par distance domicile–travail

On observe que les employés habitant très près ou relativement proches de leur lieu de travail représentent la majorité de l’effectif, 
avec près de 70 % situés à moins de 10 km.  
À l’inverse, les personnes résidant à plus de 20 km restent minoritaires (≈14 %).


In [ ]:


plt.figure(figsize=(12,7))

sns.barplot(data=df_distanceFromHome, y="Tranche_Distance",x="Pourcentage",color="skyblue")

axe = plt.gca()
axe.spines['top'].set_visible(False)
axe.spines['right'].set_visible(False)

plt.xlabel("Pourcentage (%)")
plt.ylabel("Tranche des Distances")
plt.title("Répartition des employés par distance domicile–travail (en %)")
plt.tight_layout()
plt.show(),

Une fois la distance domicile–travail analysée, j’explore maintenant la répartition des niveaux d’éducation au sein de l’entreprise.

#### 1.3.3 Niveau d’éducation
---
J’analyse ici la répartition des employés selon leur niveau d’éducation. Les niveaux codés dans le jeu de données sont convertis en catégories explicites afin de faciliter la lecture et l’interprétation.



##### Tableau des proportions

In [ ]:

education = df_vie_perso['NiveauEducation'].value_counts(normalize=True).sort_values(ascending=False)*100

df_education = education.reset_index()
df_education.columns = ["Niveau d'Education", "Pourcentage"]

df_education

##### Répartition des employés par niveau d'education

On observe qu’un peu plus de la moitié des employés ont un niveau Bachelor (≈39 %) ou Master (≈27 %).  
Environ 30 % présentent un niveau College ou Below College, tandis qu’une minorité (≈3 %) ont atteint le niveau Doctorat.

In [ ]:

plt.figure(figsize=(12,7))

axe = plt.gca()
axe.spines['top'].set_visible(False)
axe.spines['right'].set_visible(False)

sns.barplot(data=df_education, y="Niveau d'Education", x= "Pourcentage", color='skyblue')
plt.xlabel("Pourcentage")
plt.ylabel("Niveau d'Education")
plt.title("Répartition des employés par niveau d'education (en %)")
plt.tight_layout()
plt.show(),

Après avoir exploré le niveau d’éducation des employés, j’analyse maintenant leur domaine d’étude à travers la variable `EducationField`.

#### 1.3.4 Domaine d’éducation (EducationField)
---

J’analyse ici la répartition des employés selon leur domaine d’éducation (`EducationField`).  
Cette variable permet d’identifier les disciplines académiques les plus représentées dans l’entreprise et d’observer la diversité des parcours.


##### Tableau des proportions

In [ ]:
educationField = df_vie_perso["EducationField"].value_counts(normalize=True).sort_values(ascending=False)*100

df_EducationField = educationField.reset_index()
df_EducationField.columns =['Education Field', "Pourcentage"]

df_EducationField

##### Repartitions des employes par domaine d'education

CCe graphique montre que deux domaines d’éducation sont particulièrement représentés :  
- **Life Sciences** (≈ 41 %)  
- **Medical** (≈ 32 %)  

Les domaines minoritaires sont **Other** (≈ 5 %) et **Human Resources** (≈ 2 %).

Cette exploration permet d’avoir une première vision des types de parcours académiques présents dans l’entreprise et d’apprécier la diversité des profils.

In [ ]:
plt.figure(figsize=(12,7))

axe = plt.gca()
axe.spines['top'].set_visible(False)
axe.spines['right'].set_visible(False)

sns.barplot(data=df_EducationField, y= "Education Field", x= "Pourcentage", color="skyblue")
plt.xlabel("Pourcentage (%)")
plt.ylabel("Education Field")
plt.title("Répartition des employés par domaine d’éducation (en %)")
plt.tight_layout()
plt.show(),

Après l’analyse des domaines d’éducation, j’explore maintenant la répartition des employés selon leur genre.

#### 1.3.5 Genre
---
J’analyse ici la répartition des employés selon leur genre.  
Cette variable permet d’observer la composition de la population en termes de diversité et d’équilibre hommes–femmes.

##### Tableau des proportions

In [35]:
# Repartition par genre

genre = df_vie_perso["Gender"].value_counts(normalize=True).sort_values(ascending=False)*100
df_genre = genre.reset_index()

df_genre.columns = ["Genre", "Pourcentage"]
df_genre.reset_index()

df_genre

,Genre,Pourcentage
0,Homme,60.0
1,Femme,40.0


##### Repartition des employees par genre

La répartition hommes–femmes est relativement équilibrée dans l’entreprise, avec une légère majorité d'hommes représentant 60 % de l’effectif.


In [ ]:
plt.figure(figsize=(12,7))

axe = plt.gca()
axe.spines["top"].set_visible(False)
axe.spines['right'].set_visible(False)

sns.barplot(data=df_genre, y= 'Genre', x= "Pourcentage", color="skyblue")
plt.xlabel("Pourcentage")
plt.ylabel('Genre')
plt.title("Repartition des employees par genre (en %)")
plt.tight_layout()
plt.show(),

#### 

#### 1.3.6 Statut marital
---

##### Tableau des proportions

##### Repartitions des employees selon leur statut marital

#### 1.3.7 Nombre d’entreprises travaillées
---

##### Tableau des proportions

##### Repartition des employes par le nombre d'entreprise travaillés

### comparaisons simples, visualisations basiques, inspection initiale
---

Après avoir analysé individuellement les variables de l’axe "Vie personnelle", je m’intéresse maintenant à leurs liens éventuels avec l’attrition.

## 2. Attrition — Comparaisons simples
---

### croisements, graphiques, premières observations

Une fois les comparaisons simples réalisées, j’explore maintenant les relations internes à l’axe pour identifier d’éventuels patterns ou corrélations.

## 3. Relations internes à l’axe 
----

# 4. Visualisations complémentaires
---

# 5. Résultats clés (3 à 5 observations max)
---
ce que J4 observes

sans aller jusqu’à une conclusion business trop poussée